# Dashed-line detection lab: collinear grouping

Extract one page's raw vectors and split them into **layer -> color -> stroke-width** buckets. This is the P3
VectorClassification separation, but *without* the seqno merge or spatial clustering.

Within each bucket, take the **straight** Vectors: all items are `'l'`, and every point is within
`STRAIGHT_TOL_PT` of one line. Group the ones that lie on the **same infinite line**:
- angle within `ANGLE_TOL_DEG`;
- perpendicular offset within `OFFSET_TOL_PT`;
- no limit on the gap along the line.

A dashed line should show up as one group of many short vectors of similar length.

Layers (open with `scripts/pipeline_report_viewer.py`):
- `collinear / groups (distinct colour)`: each group with >= 2 members in its own hue.
- `gradient / vector count`: fewest -> most members, coloured green -> yellow -> red -> purple at
  `(n - min) / (max - min)`.
- `gradient / length std`: most -> least uniform lengths (raw pt), same colours.
- `other / singletons` and `other / excluded (non-straight)`: grey context.

Gradient ranges are page-wide and only cover groups with >= 2 members.

## 0 - Config

In [ ]:
PDF_PATH = None         # None -> first PDF under references/
PAGE = 0
ANGLE_TOL_DEG = 1.0     # single-linkage angle tolerance (folded to [0, 180))
OFFSET_TOL_PT = 0.5     # max perpendicular-offset span of one collinear group
STRAIGHT_TOL_PT = 0.25  # max point distance from the fitted line for a Vector to count as straight
LINE_WIDTH = 1.0

## 1 - Bootstrap

In [ ]:
import colorsys
import os, sys
from pathlib import Path

_root = os.path.abspath(os.path.join('../..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

import matplotlib.pyplot as plt
import numpy as np

from rastervec.commons.helpers.geometry import union_bbox
from rastervec.notebooks import _vector_probe_helpers as h

if PDF_PATH is None:
    refs = sorted((Path(_root) / 'references').glob('*.pdf'))
    assert refs, 'no PDFs found under references/'
    PDF_PATH = str(refs[0])
print(PDF_PATH, 'page', PAGE)

## 2 - Extract + layer/color/width buckets

In [ ]:
page_meta, vectors = h.load_page_vectors(PDF_PATH, PAGE)
buckets = h.bucket_vectors(vectors)
print(f'{len(vectors)} vectors -> {len(buckets)} layer/color/width buckets')
report = h.DebugReport('dashed_line_collinear_lab', PDF_PATH, page_meta, line_width=LINE_WIDTH)


def collinear_layers(summary, stage):
    """`<stage>` layer: each multi-member group in its own golden-ratio hue."""
    colour = {}
    for g, hue in zip(summary.multi, h.golden_hues(len(summary.multi))):
        rgb = colorsys.hsv_to_rgb(hue, 0.9, 0.85)
        for v, _ in g:
            colour[id(v)] = rgb
    report.add_vectors(stage, 'groups (distinct colour)', '#4a7bd0',
                       [v for g in summary.multi for v, _ in g], lambda v: colour[id(v)])


def top_groups(summary, n=15):
    rows = sorted(zip(summary.multi, summary.counts, summary.stds), key=lambda r: -r[1])[:n]
    print(f"{'n':>5} {'angle':>7} {'mean len':>9} {'len std':>8}  dashes attr")
    for g, count, std in rows:
        angle = h._mean_axis_angle([fit.angle for _, fit in g])
        mean_len = np.mean([fit.length for _, fit in g])
        print(f'{count:>5} {angle:>7.2f} {mean_len:>9.2f} {std:>8.3f}  {g[0][0].dashes}')

## 3 - Collinear straight Vectors within each bucket

In [ ]:
v_groups, v_excluded = [], []
for key, members in buckets.items():
    straight, rest = h.split_straight(members, STRAIGHT_TOL_PT)
    v_excluded.extend(rest)
    v_groups.extend(h.group_collinear(straight, ANGLE_TOL_DEG, OFFSET_TOL_PT))
v_summary = h.summarise_groups(v_groups)
print(v_summary.describe(len(v_excluded), 'non-straight vector(s)'))

collinear_layers(v_summary, 'collinear')
h.add_gradient_layers(report, v_summary, v_excluded, excluded_label='excluded (non-straight)')
fig = h.group_histograms(v_summary, 'Collinear')
report.save_figure(fig, 'histograms')
plt.show()
top_groups(v_summary)

## 4 - Write manifest

In [ ]:
out = report.finish()
print('wrote', out)
for e in report.layers:
    print(f"  {e['stage']:>14} | {e['layer']}")
print()
print('open with:')
print(report.viewer_command())